In [ ]:
import os

import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow import (
    log_metric,
    log_params,
    start_run,
)
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest as IsoF
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from skopt import BayesSearchCV
from skopt.space import Categorical, Integer

import dataprocessing as dc


In [2]:
df = pd.read_csv("data/email_data.csv")
newdata = dc.Data_class(data = df, )
df = newdata.data
df["Category"] = np.where(df['Category'] == 'spam', -1, 1)


In [ ]:
y = df["Category"]
X = df.drop(
    ["Category", "Message", "word_count"],
    axis = 1
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    stratify= y,
    random_state = 42
)





In [5]:
numeric_cols = X.select_dtypes(
    include = ["number"]
).columns
numeric_pipe = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="mean")),
        ("encoder",MinMaxScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, numeric_cols)
    ],
    remainder='drop'  # or 'passthrough' if you want to keep unlisted columns
)

In [6]:
model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("model", IsoF(random_state = 42))
    ]
)

In [ ]:


#determining number of features max
n_features = X_train.shape[1]

#determing rate of spam within data
spam_rate = float((y_train == -1).mean())

search_space = {
    "model__n_estimators": Integer(100, 500),
    "model__max_samples":  Categorical(['auto', 64, 128, 256, 512]),
    "model__max_features": Categorical([
        1.0,
        0.5,
        0.75,
        int(n_features ** 0.5),
        int(np.log2(n_features)),
    ]),
    "model__contamination": Categorical([
        "auto",
        spam_rate,                    
        spam_rate * 0.5,              
        spam_rate * 1.5,              
        0.1,
        0.2,
        0.3,
    ]),
}

In [ ]:
spam_scorer = make_scorer(f1_score, pos_label=-1)

#%%capture #uncomment to suppress output
with start_run(
    run_name="IsolationForest_BayesianSearch_AllConfigs",
    log_system_metrics=True,
):

    bayes_search_cv = BayesSearchCV(
        estimator=model,
        search_spaces=search_space,
        n_iter=128,
        cv=5,
        scoring=spam_scorer,
        random_state=42,
        verbose=0,
    )
    bayes_search_cv.fit(X_train, y_train) 

    log_params(bayes_search_cv.best_params_)
    log_metric("cv_best_spam_f1", bayes_search_cv.best_score_)
    mlflow.sklearn.log_model(
        bayes_search_cv.best_estimator_,
        name="best_model",
        serialization_format="cloudpickle",
    )

    for i, candidate in enumerate(bayes_search_cv.cv_results_["params"]):
        with mlflow.start_run(nested=True, run_name=f"CV_Set_{i}"):
            log_params(candidate)


            log_metric("cv_mean_spam_f1", bayes_search_cv.cv_results_["mean_test_score"][i])
            log_metric("cv_std_spam_f1",  bayes_search_cv.cv_results_["std_test_score"][i])


            m = clone(bayes_search_cv.estimator).set_params(**candidate)
            m.fit(X_train, y_train)


            y_pred  = m.predict(X_test)
            y_score = -m.score_samples(X_test)


            y_bin = (y_test == -1).astype(int)

            # metrics
            log_metric("test_roc_auc", roc_auc_score(y_bin, y_score))
            log_metric("test_accuracy", accuracy_score(y_test, y_pred))
            log_metric("precision", precision_score(y_test, y_pred, pos_label=-1, zero_division=0))
            log_metric("recall", recall_score(y_test, y_pred, pos_label=-1, zero_division=0))
            log_metric("f1 score", f1_score(y_test, y_pred, pos_label=-1, zero_division=0))
            log_metric("weighted f1 score", f1_score(y_test, y_pred, average="weighted"))

2026/09/25 16:05:23 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/09/25 16:05:23 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


/home/michael/bda602/.venv/lib/python3.14/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.float64(0.2012564505272605), np.float64(0.75), np.int64(64), np.int64(100)] before, using random point [0.13417096701817366, 1.0, 'auto', np.int64(453)]
  warnings.warn(
/home/michael/bda602/.venv/lib/python3.14/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.float64(0.2012564505272605), np.int64(7), np.int64(64), np.int64(500)] before, using random point [0.2012564505272605, 7, 'auto', np.int64(456)]
  warnings.warn(
/home/michael/bda602/.venv/lib/python3.14/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.float64(0.2012564505272605), np.float64(1.0), np.int64(64), np.int64(500)] before, using random point [0.13417096701817366, 0.75, 128, np.int64(347)]
  warnings.warn(
/home/michael/bda602/.venv/lib/python3.14/site-package

In [11]:
print("Best Parameters:", bayes_search_cv.best_params_)
print("Best Score:", bayes_search_cv.best_score_)

# Predict with best model
y_pred = bayes_search_cv.best_estimator_.predict(X_test)

Best Parameters: OrderedDict({'model__contamination': 0.2012564505272605, 'model__max_features': 7, 'model__max_samples': 64, 'model__n_estimators': 100})
Best Score: 0.46286180101832
